# AMD Scalper — Validation & Backtesting
### Load Checkpoint → Evaluate → Realistic Backtest ($100 start, 10% risk, OANDA conditions)

**AMD Scalper** is a 5-minute session manipulation reversal model using the AMD (Accumulation–Manipulation–Distribution) framework with 3 gated entry models (S&R, S&D, ORB).

This notebook:
1. Loads trained AMD Scalper checkpoint (from Google Drive or local)
2. Runs holdout test evaluation — signal accuracy, phase accuracy, gate analysis
3. Backtests with **realistic OANDA conditions**: $100 start, 10% risk, real spreads, 30:1 leverage
4. Full trade breakdown: daily/weekly/monthly, session analysis (critical for AMD)
5. Realistic friction simulation with OANDA-specific parameters

---
| OANDA Reality | Value |
|---------------|-------|
| Starting Balance | $100 |
| Risk per Trade | 10% ($10 at start) |
| Leverage | 30:1 (ESMA retail) |
| GBP/JPY spread | 2.5 pips (OANDA avg) |
| EUR/JPY spread | 2.0 pips |
| GBP/USD spread | 1.6 pips |
| Commission | $0 (OANDA Standard) |
| Min lot | 0.01 (1,000 units) |
| OANDA Account | `101-001-30902818-003` |

## 1. Setup & Mount Google Drive

In [ ]:
# ── Workspace Setup ───────────────────────────────────────────────────────────
from google.colab import drive
import os, pathlib, sys

drive.mount("/content/drive")

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/phase_lm")
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
DRIVE_PROC = DRIVE_ROOT / "processed_data"

LOCAL_ROOT = pathlib.Path("/content/phase_lm")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted.")
print(f"Checkpoints dir  : {DRIVE_CKPT}")
print(f"Processed data   : {DRIVE_PROC}")

# Show available AMD Scalper checkpoints
amd_ckpts = sorted(DRIVE_CKPT.glob("amd_scalper_*"), key=lambda p: p.name, reverse=True)
print(f"\nAMD Scalper checkpoints found: {len(amd_ckpts)}")
for c in amd_ckpts[:5]:
    print(f"  {c.name}")

## 2. Clone Repository & Install Dependencies

In [ ]:
import subprocess, shutil

REPO_URL = "https://github.com/Unseengap/wavetrader.git"
REPO_DIR = "/content/phase_lm"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Cloning repository...")
    if os.path.exists(REPO_DIR):
        os.chdir("/content")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository exists. Pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "stash"], check=False)
    subprocess.run(["git", "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ta", "pandas_ta", "scikit-learn", "scipy",
                "fastparquet", "pyarrow"], check=False)
if os.path.exists("requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete.")

## 3. Load AMD Scalper Model from Checkpoint

In [ ]:
import torch, json, hashlib
from wavetrader.amd_scalper import AMDScalper
from wavetrader.config import AMDScalperConfig

# ── Select checkpoint (latest amd_scalper_* dir) ──────────────────────────────
amd_dirs = sorted(DRIVE_CKPT.glob("amd_scalper_*"), key=lambda p: p.name, reverse=True)
if not amd_dirs:
    raise RuntimeError(f"No AMD Scalper checkpoints in {DRIVE_CKPT}")

CKPT_DIR = amd_dirs[0]
print(f"Loading checkpoint: {CKPT_DIR.name}")

# ── Verify SHA-256 integrity ───────────────────────────────────────────────────
weights_path = CKPT_DIR / "model_weights.pt"
sha_path = CKPT_DIR / "weights.sha256"

if sha_path.exists():
    expected = sha_path.read_text().split()[0]
    h = hashlib.sha256()
    with open(weights_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    actual = h.hexdigest()
    if expected == actual:
        print(f"SHA-256 verified: {actual[:24]}...")
    else:
        raise RuntimeError(f"Checksum MISMATCH! expected={expected[:24]}, actual={actual[:24]}")

# ── Load config ────────────────────────────────────────────────────────────────
config_path = CKPT_DIR / "config.json"
if config_path.exists():
    with open(config_path) as f:
        cfg_dict = json.load(f)
    clean = {k: v for k, v in cfg_dict.items()
             if not k.startswith("_") and k in AMDScalperConfig.__dataclass_fields__}
    config = AMDScalperConfig(**clean)
    print(f"Config loaded: {config.timeframes}, fused_dim={config.fused_dim}")
else:
    config = AMDScalperConfig()
    print("No config.json found — using defaults")

# ── Instantiate & load weights ─────────────────────────────────────────────────
model = AMDScalper(config).to(device)

checkpoint = torch.load(str(weights_path), map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# Strip _orig_mod. prefix from torch.compile'd checkpoints
cleaned = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(cleaned)

if isinstance(checkpoint, dict):
    train_info = checkpoint.get("training", {})
    print(f"Best val acc: {train_info.get('best_val_accuracy', 'N/A')}")
    print(f"Test acc    : {train_info.get('test_accuracy', 'N/A')}")
    print(f"Gate avg    : {train_info.get('gate_avg', 'N/A')}")

model.eval()
print(f"\nAMD Scalper loaded: {model.count_parameters():,} params on {device}")

## 4. Load Test Data (5-minute from 1-minute resampling + higher TFs)

In [ ]:
import time
import numpy as np
import pandas as pd
import torch.utils.data as tud
from wavetrader.amd_dataset import AMDForexDataset, amd_collate_fn
from wavetrader.data import _normalise_df, _resample_ohlcv
from torch.utils.data import ConcatDataset

PAIRS = ["GBPJPY", "EURJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY", "GBPUSD": "GBP/USD"}
TFS_HIGHER = ["15m", "1h", "4h"]
TF_KEY = {"15m": "15min", "1h": "1h", "4h": "4h"}

# OANDA typical spreads per pair (pips)
OANDA_SPREADS = {"GBPJPY": 2.5, "EURJPY": 2.0, "GBPUSD": 1.6}
PIP_VALUES = {"GBPJPY": 6.5, "EURJPY": 6.5, "GBPUSD": 10.0}

LOOKAHEAD = 6
THRESHOLD_PIPS = 15.0


def load_5min_test(pair_tag, drive_root):
    """Load 1-minute raw CSVs from Drive, resample to 5min, take test split."""
    raw_dir = drive_root / "data"

    # Try ask+bid mid-price
    ask_files = sorted(raw_dir.glob(f"{pair_tag}*1 Min*Ask*.csv"))
    bid_files = sorted(raw_dir.glob(f"{pair_tag}*1 Min*Bid*.csv"))

    if ask_files and bid_files:
        ask_df = _normalise_df(pd.read_csv(ask_files[0]))
        bid_df = _normalise_df(pd.read_csv(bid_files[0]))
        merged = ask_df.merge(bid_df, on="date", suffixes=("_ask", "_bid"))
        mid = pd.DataFrame({
            "date": merged["date"],
            "open": (merged["open_ask"] + merged["open_bid"]) / 2,
            "high": (merged["high_ask"] + merged["high_bid"]) / 2,
            "low": (merged["low_ask"] + merged["low_bid"]) / 2,
            "close": (merged["close_ask"] + merged["close_bid"]) / 2,
            "volume": merged.get("volume_ask", merged.get("volume_bid", 0)),
        })
    else:
        one_min = sorted(raw_dir.glob(f"{pair_tag}*1 Min*.csv"))
        if not one_min:
            return None
        mid = _normalise_df(pd.read_csv(one_min[0]))

    df_5m = _resample_ohlcv(mid, "5min")
    # Take last 15% as test (matches training split)
    n = len(df_5m)
    test_start = int(n * 0.85)
    return df_5m.iloc[test_start:].reset_index(drop=True)


def load_higher_tf_test(pair_tag, tf_short, drive_proc):
    """Load higher TF test parquet."""
    path = drive_proc / "test" / f"{pair_tag}_{tf_short}.parquet"
    if not path.exists():
        return None
    df = pd.read_parquet(path)
    df = _normalise_df(df)
    return df


# Load all test data
test_data = {}
test_datasets = []
per_pair_datasets = {}

print("Loading test data...")
for pair_tag in PAIRS:
    dfs = {}

    # 5min from 1min resampling (test split)
    df_5m = load_5min_test(pair_tag, DRIVE_ROOT)
    if df_5m is None:
        print(f"  {pair_tag}: SKIPPED (no 1min data)")
        continue
    dfs["5min"] = df_5m
    print(f"  {pair_tag} 5min: {len(df_5m):,} bars")

    # Higher TFs
    for tf_short in TFS_HIGHER:
        df_tf = load_higher_tf_test(pair_tag, tf_short, DRIVE_PROC)
        if df_tf is not None:
            dfs[TF_KEY[tf_short]] = df_tf
            print(f"  {pair_tag} {tf_short}: {len(df_tf):,} bars")

    if len(dfs) == len(config.timeframes):
        test_data[pair_tag] = dfs
        ds = AMDForexDataset(dfs, config, lookahead=LOOKAHEAD,
                             threshold_pips=THRESHOLD_PIPS, pair=PAIR_NAMES[pair_tag])
        test_datasets.append(ds)
        per_pair_datasets[pair_tag] = ds
        print(f"  {pair_tag}: {len(ds):,} samples ✓")
    else:
        print(f"  {pair_tag}: only {len(dfs)}/{len(config.timeframes)} TFs — skipping")

# Combined dataset
combined_test = ConcatDataset(test_datasets)
test_loader = tud.DataLoader(
    combined_test, batch_size=128, shuffle=False,
    collate_fn=amd_collate_fn, num_workers=2, pin_memory=True,
)

print(f"\nCombined test: {len(combined_test):,} samples, {len(test_loader)} batches")

## 5. Holdout Validation — Signal Accuracy, Phase Accuracy, Gate Analysis

In [ ]:
# ── Holdout Evaluation: Signal + Phase + Gate ─────────────────────────────────
from wavetrader.train_amd_scalper import AMDScalperLoss
from collections import Counter
from torch.amp import autocast

criterion = AMDScalperLoss().to(device)
model.eval()

total_loss, correct, n = 0.0, 0, 0
all_preds, all_labels = [], []
all_phase_preds, all_phase_labels = [], []
gate_sums = torch.zeros(3)
gate_count = 0
all_confs = []

SIGNAL_NAMES = ["BUY", "SELL", "HOLD"]
PHASE_NAMES = ["ACCUM", "MANIP", "DIST", "INVALID"]
num_batches = len(test_loader)
use_amp = device.type == "cuda"

print(f"Evaluating AMD Scalper on combined test set ({num_batches} batches)...")
t0 = time.time()

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        labels = batch["label"].to(device)
        phase_labels = batch.get("phase_label")
        if phase_labels is not None:
            phase_labels = phase_labels.to(device)

        model_input = {
            tf: {k: v.to(device) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }

        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, labels, phase_labels)

        total_loss += losses["total"].item()
        preds = out["signal_logits"].argmax(-1)
        correct += (preds == labels).sum().item()
        n += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_confs.extend(out["confidence"].squeeze(-1).cpu().tolist())

        # Phase predictions
        if phase_labels is not None:
            phase_preds = out["phase_logits"].argmax(-1)
            all_phase_preds.extend(phase_preds.cpu().tolist())
            all_phase_labels.extend(phase_labels.cpu().tolist())

        # Gate usage
        gate_sums += out["gate_weights"].mean(0).cpu()
        gate_count += 1

        if (i + 1) % max(1, num_batches // 5) == 0 or (i + 1) == num_batches:
            print(f"  [{(i+1)/num_batches*100:5.1f}%]  {n:,} samples  |  "
                  f"acc: {correct/max(n,1)*100:.2f}%  |  {time.time()-t0:.1f}s")

val_loss = total_loss / max(num_batches, 1)
val_acc = correct / max(n, 1)
avg_gate = gate_sums / max(gate_count, 1)

print(f"\n{'='*60}")
print(f"  AMD Scalper Holdout Results")
print(f"{'='*60}")
print(f"  Loss     : {val_loss:.4f}")
print(f"  Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"  Samples  : {n:,}")
print(f"  Gate avg : S&R={avg_gate[0]:.3f}, S&D={avg_gate[1]:.3f}, ORB={avg_gate[2]:.3f}")
print(f"  Avg conf : {np.mean(all_confs):.3f} (std={np.std(all_confs):.3f})")

# Per-class signal breakdown
print(f"\n  Signal Class Breakdown:")
label_counts = Counter(all_labels)
pred_counts = Counter(all_preds)
for i, name in enumerate(SIGNAL_NAMES):
    n_true = label_counts.get(i, 0)
    n_pred = pred_counts.get(i, 0)
    cc = sum(1 for p, l in zip(all_preds, all_labels) if l == i and p == i)
    ca = cc / max(n_true, 1)
    print(f"    {name:5s}: true={n_true:5d}, pred={n_pred:5d}, acc={ca:.2%}")

# Phase accuracy
if all_phase_preds:
    phase_correct = sum(1 for p, l in zip(all_phase_preds, all_phase_labels) if p == l)
    phase_acc = phase_correct / max(len(all_phase_preds), 1)
    print(f"\n  Phase Classification Accuracy: {phase_acc:.2%}")
    phase_label_counts = Counter(all_phase_labels)
    phase_pred_counts = Counter(all_phase_preds)
    for i, name in enumerate(PHASE_NAMES):
        n_true = phase_label_counts.get(i, 0)
        n_pred = phase_pred_counts.get(i, 0)
        cc = sum(1 for p, l in zip(all_phase_preds, all_phase_labels) if l == i and p == i)
        ca = cc / max(n_true, 1)
        print(f"    {name:8s}: true={n_true:5d}, pred={n_pred:5d}, acc={ca:.2%}")

# Confusion matrix
print(f"\n  Signal Confusion Matrix:")
print(f"  {'':>10} {'BUY':>8} {'SELL':>8} {'HOLD':>8}")
for i, name in enumerate(SIGNAL_NAMES):
    row = [sum(1 for p, l in zip(all_preds, all_labels) if l == i and p == j) for j in range(3)]
    print(f"  {name:>10} {row[0]:8d} {row[1]:8d} {row[2]:8d}")

# BUY/SELL-only accuracy
trade_preds = [(p, l) for p, l in zip(all_preds, all_labels) if l in (0, 1)]
if trade_preds:
    trade_acc = sum(1 for p, l in trade_preds if p == l) / len(trade_preds)
    print(f"\n  BUY/SELL accuracy (excl HOLD): {trade_acc:.2%} ({len(trade_preds)} samples)")
print(f"{'='*60}")

## 6. Realistic Backtest — $100 Start, 10% Risk, OANDA Conditions

Runs a separate backtest per pair with pair-specific OANDA spreads and pip values.

In [ ]:
from wavetrader.backtest import run_backtest
from wavetrader.config import BacktestConfig

INITIAL_BALANCE = 100.0
RISK_PER_TRADE = 0.10

all_results = {}

for pair_tag in test_data:
    pair_name = PAIR_NAMES[pair_tag]
    spread = OANDA_SPREADS.get(pair_tag, 2.5)
    pip_val = PIP_VALUES.get(pair_tag, 6.5)

    bt_config = BacktestConfig(
        initial_balance=INITIAL_BALANCE,
        risk_per_trade=RISK_PER_TRADE,
        min_confidence=0.55,
        spread_pips=spread,
        pip_value=pip_val,
        commission_per_lot=0.0,
        leverage=30.0,
        atr_halt_multiplier=3.0,
        drawdown_reduce_threshold=0.15,
    )

    bt_model_config = AMDScalperConfig(
        timeframes=config.timeframes,
        lookbacks=config.lookbacks,
        tf_wave_dim=config.tf_wave_dim,
        fused_dim=config.fused_dim,
        predictor_layers=config.predictor_layers,
        predictor_heads=config.predictor_heads,
        dropout=config.dropout,
    )

    print(f"\n{'='*60}")
    print(f"  Backtesting {pair_name} (spread={spread} pips, pip_val=${pip_val})")
    print(f"{'='*60}")

    results = run_backtest(model, test_data[pair_tag], bt_model_config, bt_config, device)
    all_results[pair_tag] = (results, bt_config)

# ── Aggregate summary ─────────────────────────────────────────────────────────
print(f"\n\n{'='*75}")
print(f"  AGGREGATE BACKTEST — AMD Scalper ($100 start, 10% risk, OANDA)")
print(f"{'='*75}")
print(f"{'Pair':>10} | {'Trades':>7} | {'Win Rate':>8} | {'PF':>6} | "
      f"{'Final $':>10} | {'ROI':>8} | {'MaxDD':>7} | {'Sharpe':>7}")
print(f"{'-'*75}")

total_trades = 0
for pair_tag in all_results:
    r, bc = all_results[pair_tag]
    roi = (r.final_balance / bc.initial_balance - 1) * 100
    print(f"{PAIR_NAMES[pair_tag]:>10} | {r.total_trades:>7} | {r.win_rate:>7.1%} | "
          f"{r.profit_factor:>6.2f} | ${r.final_balance:>9,.2f} | "
          f"{roi:>+7.1f}% | {r.max_drawdown:>6.1%} | {r.sharpe_ratio:>7.2f}")
    total_trades += r.total_trades

print(f"{'-'*75}")
print(f"Total trades across all pairs: {total_trades}")

## 7. Primary Pair Deep Dive — Trade Breakdown, Period Analysis

In [ ]:
# ── Select primary pair for deep analysis ──────────────────────────────────────
PRIMARY_PAIR = "GBPJPY"
if PRIMARY_PAIR not in all_results:
    PRIMARY_PAIR = list(all_results.keys())[0]

results, bt_config = all_results[PRIMARY_PAIR]
trades = results.trades
pair_name = PAIR_NAMES[PRIMARY_PAIR]

print(f"Deep dive: {pair_name} — {len(trades)} closed trades")

# Build trade DataFrame
rows = []
for t in trades:
    direction = "BUY" if t.direction.value == 0 else "SELL"
    rows.append({
        "entry_time": t.entry_time, "exit_time": t.exit_time,
        "direction": direction, "entry_price": t.entry_price,
        "exit_price": t.exit_price, "stop_loss": t.stop_loss,
        "take_profit": t.take_profit, "size_lots": t.size,
        "pnl": t.pnl, "exit_reason": t.exit_reason,
        "is_winner": t.pnl > 0,
    })

trade_df = pd.DataFrame(rows)
trade_df["entry_time"] = pd.to_datetime(trade_df["entry_time"])
trade_df["exit_time"] = pd.to_datetime(trade_df["exit_time"])
trade_df["cumulative_pnl"] = trade_df["pnl"].cumsum()
trade_df["trade_num"] = range(1, len(trade_df) + 1)
trade_df["duration"] = trade_df["exit_time"] - trade_df["entry_time"]
trade_df["balance_after"] = bt_config.initial_balance + trade_df["cumulative_pnl"]
trade_df["running_win_rate"] = trade_df["is_winner"].expanding().mean()

print(f"Date range: {trade_df['entry_time'].min()} → {trade_df['exit_time'].max()}")
print(f"\n── First 5 Trades ──")
display(trade_df.head())

# ── Period Breakdown ──────────────────────────────────────────────────────────
def period_stats(group):
    wins = group["is_winner"].sum()
    losses = len(group) - wins
    gp = group.loc[group["pnl"] > 0, "pnl"].sum()
    gl = abs(group.loc[group["pnl"] <= 0, "pnl"].sum())
    return pd.Series({
        "trades": len(group), "wins": int(wins), "losses": int(losses),
        "win_rate": wins / max(len(group), 1),
        "gross_profit": gp, "gross_loss": gl,
        "net_pnl": group["pnl"].sum(), "avg_pnl": group["pnl"].mean(),
        "max_win": group["pnl"].max(), "max_loss": group["pnl"].min(),
        "profit_factor": gp / max(gl, 1e-9),
        "avg_duration": group["duration"].mean(),
    })

trade_df["exit_date"] = trade_df["exit_time"].dt.date
daily = trade_df.groupby("exit_date").apply(period_stats).reset_index()
daily["exit_date"] = pd.to_datetime(daily["exit_date"])
daily["cumulative_pnl"] = daily["net_pnl"].cumsum()

trade_df["exit_week"] = trade_df["exit_time"].dt.to_period("W")
weekly = trade_df.groupby("exit_week").apply(period_stats).reset_index()
weekly["cumulative_pnl"] = weekly["net_pnl"].cumsum()

trade_df["exit_month"] = trade_df["exit_time"].dt.to_period("M")
monthly = trade_df.groupby("exit_month").apply(period_stats).reset_index()
monthly["cumulative_pnl"] = monthly["net_pnl"].cumsum()

trade_df["exit_year"] = trade_df["exit_time"].dt.year
yearly = trade_df.groupby("exit_year").apply(period_stats).reset_index()
yearly["cumulative_pnl"] = yearly["net_pnl"].cumsum()

print(f"\n{'='*80}")
print(f"  YEARLY BREAKDOWN — {pair_name}")
print(f"{'='*80}")
for _, r in yearly.iterrows():
    print(f"  {int(r['exit_year'])}  |  Trades: {int(r['trades']):>5}  |  "
          f"W/L: {int(r['wins'])}/{int(r['losses'])}  |  WR: {r['win_rate']:.1%}  |  "
          f"PnL: ${r['net_pnl']:>+12,.2f}  |  PF: {r['profit_factor']:.2f}")

print(f"\n{'='*80}")
print(f"  MONTHLY BREAKDOWN")
print(f"{'='*80}")
for _, r in monthly.iterrows():
    print(f"  {str(r['exit_month']):>7}  |  Trades: {int(r['trades']):>4}  |  "
          f"W/L: {int(r['wins']):>3}/{int(r['losses']):<3}  |  WR: {r['win_rate']:.0%}  |  "
          f"PnL: ${r['net_pnl']:>+10,.2f}  |  PF: {r['profit_factor']:>5.2f}")

print(f"\n{'='*80}")
print(f"  WEEKLY SUMMARY ({len(weekly)} weeks)")
print(f"{'='*80}")
print(f"  Profitable weeks : {(weekly['net_pnl'] > 0).sum()} / {len(weekly)}  "
      f"({(weekly['net_pnl'] > 0).mean():.1%})")
print(f"  Best week PnL    : ${weekly['net_pnl'].max():>+12,.2f}")
print(f"  Worst week PnL   : ${weekly['net_pnl'].min():>+12,.2f}")
print(f"  Avg week PnL     : ${weekly['net_pnl'].mean():>+12,.2f}")

print(f"\n{'='*80}")
print(f"  DAILY SUMMARY ({len(daily)} trading days)")
print(f"{'='*80}")
print(f"  Profitable days  : {(daily['net_pnl'] > 0).sum()} / {len(daily)}  "
      f"({(daily['net_pnl'] > 0).mean():.1%})")
print(f"  Best day PnL     : ${daily['net_pnl'].max():>+12,.2f}")
print(f"  Worst day PnL    : ${daily['net_pnl'].min():>+12,.2f}")
print(f"  Avg trades/day   : {daily['trades'].mean():.1f}")

## 8. Timeline Visualizations & Direction Analysis

In [ ]:
# ── Timeline Visualizations ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(16, 20))
fig.suptitle(f"AMD Scalper Trade Analysis — {pair_name} ($100 start, 10% risk)",
             fontsize=16, fontweight="bold", y=0.98)

# 1. Per-trade PnL + cumulative balance
ax = axes[0]
colors = ["#4CAF50" if p > 0 else "#F44336" for p in trade_df["pnl"]]
ax.bar(trade_df["trade_num"], trade_df["pnl"], color=colors, width=1.0, alpha=0.5, label="Per-trade PnL")
ax2 = ax.twinx()
ax2.plot(trade_df["trade_num"], trade_df["balance_after"], color="#2196F3", lw=1.5, label="Balance")
ax2.set_ylabel("Balance ($)")
ax.set_xlabel("Trade #")
ax.set_ylabel("PnL ($)")
ax.set_title("Per-Trade PnL & Balance Growth")
ax.grid(alpha=0.2)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# 2. Monthly PnL bars
ax = axes[1]
month_labels = [str(m) for m in monthly["exit_month"]]
monthly_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in monthly["net_pnl"]]
ax.bar(range(len(monthly)), monthly["net_pnl"], color=monthly_colors, edgecolor="white", lw=0.5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(month_labels, rotation=45, ha="right", fontsize=8)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("Net PnL ($)")
ax.set_title("Monthly Net PnL")
ax.grid(alpha=0.2, axis="y")

# 3. Rolling win rate
ax = axes[2]
window = min(50, len(trade_df) // 5) or 10
rolling_wr = trade_df["is_winner"].rolling(window).mean() * 100
ax.plot(trade_df["trade_num"], rolling_wr, color="#FF9800", lw=1.2, label=f"{window}-trade WR")
ax.axhline(50, linestyle="--", color="gray", alpha=0.6)
ax.fill_between(trade_df["trade_num"], 50, rolling_wr,
                where=rolling_wr >= 50, color="#4CAF50", alpha=0.15)
ax.fill_between(trade_df["trade_num"], 50, rolling_wr,
                where=rolling_wr < 50, color="#F44336", alpha=0.15)
ax.set_xlabel("Trade #")
ax.set_ylabel("Win Rate (%)")
ax.set_title(f"Rolling Win Rate ({window}-trade window)")
ax.legend()
ax.grid(alpha=0.2)
ax.set_ylim(0, 100)

# 4. PnL by exit reason
ax = axes[3]
reason_stats = trade_df.groupby("exit_reason").agg(
    count=("pnl", "size"), total_pnl=("pnl", "sum"),
    win_rate=("is_winner", "mean"),
).sort_values("count", ascending=True)
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in reason_stats["total_pnl"]]
y = range(len(reason_stats))
ax.barh(y, reason_stats["total_pnl"], color=bar_colors, edgecolor="white")
ax.set_yticks(y)
ax.set_yticklabels([f"{r}\n({int(c)} trades, {wr:.0%})" for r, c, wr
                    in zip(reason_stats.index, reason_stats["count"], reason_stats["win_rate"])])
ax.set_xlabel("Total PnL ($)")
ax.set_title("PnL by Exit Reason")
ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.2, axis="x")

plt.tight_layout()
plt.show()

# Duration distribution (critical for scalper — should be short)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dur_mins = trade_df["duration"].dt.total_seconds() / 60
axes[0].hist(dur_mins, bins=50, color="#9C27B0", alpha=0.7, edgecolor="white")
axes[0].set_xlabel("Duration (minutes)")
axes[0].set_title(f"Trade Duration Distribution (median={dur_mins.median():.0f}min)")
axes[0].grid(alpha=0.2)

win_dur = dur_mins[trade_df["is_winner"]]
loss_dur = dur_mins[~trade_df["is_winner"]]
axes[1].hist(win_dur, bins=40, alpha=0.6, color="#4CAF50", label=f"Winners ({win_dur.mean():.0f}min)")
axes[1].hist(loss_dur, bins=40, alpha=0.6, color="#F44336", label=f"Losers ({loss_dur.mean():.0f}min)")
axes[1].set_xlabel("Duration (minutes)")
axes[1].set_title("Duration: Winners vs Losers")
axes[1].legend()
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Direction breakdown + Streak analysis
print(f"\n{'='*70}")
print(f"  DIRECTION BREAKDOWN — {pair_name}")
print(f"{'='*70}")
for direction in ["BUY", "SELL"]:
    d = trade_df[trade_df["direction"] == direction]
    if len(d) == 0: continue
    wins = d["is_winner"].sum()
    gp = d.loc[d["pnl"] > 0, "pnl"].sum()
    gl = abs(d.loc[d["pnl"] <= 0, "pnl"].sum())
    print(f"  {direction:>4}  |  Trades: {len(d):>5}  |  W/L: {int(wins)}/{len(d)-int(wins)}  |  "
          f"WR: {wins/len(d):.1%}  |  Net: ${d['pnl'].sum():>+12,.2f}  |  PF: {gp/max(gl,1e-9):.2f}")

streaks = []
cs = 0
for w in trade_df["is_winner"]:
    cs = max(0, cs) + 1 if w else min(0, cs) - 1
    streaks.append(cs)
trade_df["streak"] = streaks
print(f"\n  Max winning streak : {max(streaks)} trades")
print(f"  Max losing streak  : {abs(min(streaks))} trades")

## 9. Session & Time-of-Day Analysis (Critical for AMD)

The AMD framework is session-dependent: accumulation in Asia, manipulation in London, distribution in NY. This analysis validates whether the model is actually trading during the expected NY distribution phase.

In [ ]:
# ── Session & Time-of-Day Analysis ─────────────────────────────────────────────
trade_df["entry_hour"] = trade_df["entry_time"].dt.hour
trade_df["entry_dow"] = trade_df["entry_time"].dt.day_name()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"AMD Scalper Session Analysis — {pair_name}", fontsize=14, fontweight="bold")

# 1. Trades by hour — with session bands
ax = axes[0, 0]
hourly = trade_df.groupby("entry_hour").agg(
    trades=("pnl", "size"), net_pnl=("pnl", "sum"), win_rate=("is_winner", "mean"),
).reindex(range(24), fill_value=0)
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in hourly["net_pnl"]]
ax.bar(hourly.index, hourly["trades"], color=bar_colors, edgecolor="white", alpha=0.8)
# Session bands (UTC)
ax.axvspan(0, 9, alpha=0.08, color="#FF9800", label="Asian (Accum)")
ax.axvspan(7, 13, alpha=0.08, color="#F44336", label="London (Manip)")
ax.axvspan(13, 22, alpha=0.08, color="#4CAF50", label="NY (Dist)")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("# Trades")
ax.set_title("Trades by Hour — AMD Session Overlay")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.2)

# 2. PnL by hour
ax = axes[0, 1]
pnl_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in hourly["net_pnl"]]
ax.bar(hourly.index, hourly["net_pnl"], color=pnl_colors, edgecolor="white", alpha=0.8)
ax.axvspan(13, 22, alpha=0.08, color="#4CAF50")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Net PnL ($)")
ax.set_title("PnL by Hour (NY session = green band)")
ax.grid(alpha=0.2)

# 3. Win rate by hour
ax = axes[1, 0]
wr_data = hourly[hourly["trades"] > 0]["win_rate"] * 100
ax.bar(wr_data.index, wr_data, color="#2196F3", edgecolor="white", alpha=0.8)
ax.axhline(50, linestyle="--", color="gray", alpha=0.6)
ax.axvspan(13, 22, alpha=0.08, color="#4CAF50")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Win Rate (%)")
ax.set_title("Win Rate by Hour")
ax.grid(alpha=0.2)
ax.set_ylim(0, 100)

# 4. PnL by day of week
ax = axes[1, 1]
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow_stats = trade_df.groupby("entry_dow").agg(
    trades=("pnl", "size"), net_pnl=("pnl", "sum"), win_rate=("is_winner", "mean"),
)
dow_stats = dow_stats.reindex(dow_order)
dow_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in dow_stats["net_pnl"]]
ax.bar(range(len(dow_stats)), dow_stats["net_pnl"], color=dow_colors, edgecolor="white")
ax.set_xticks(range(len(dow_stats)))
ax.set_xticklabels([f"{d[:3]}\n({int(t)} trades)" for d, t in
                    zip(dow_stats.index, dow_stats["trades"])], fontsize=9)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("Net PnL ($)")
ax.set_title("PnL by Day of Week")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Session breakdown table
def _session(hour):
    if 0 <= hour < 9: return "Asian"
    if 7 <= hour < 13: return "London"  # overlap
    if 13 <= hour < 22: return "New York"
    return "Off-hours"

trade_df["session"] = trade_df["entry_hour"].apply(_session)
session_stats = trade_df.groupby("session").agg(
    trades=("pnl", "size"), wins=("is_winner", "sum"),
    net_pnl=("pnl", "sum"), avg_pnl=("pnl", "mean"),
    avg_dur_min=("duration", lambda x: x.mean().total_seconds() / 60),
)
session_stats["win_rate"] = session_stats["wins"] / session_stats["trades"]
session_stats["gp"] = trade_df.groupby("session").apply(
    lambda g: g.loc[g["pnl"] > 0, "pnl"].sum()
)
session_stats["gl"] = trade_df.groupby("session").apply(
    lambda g: abs(g.loc[g["pnl"] <= 0, "pnl"].sum())
)
session_stats["profit_factor"] = session_stats["gp"] / session_stats["gl"].clip(lower=1e-9)

print(f"\n{'='*80}")
print(f"  SESSION BREAKDOWN — {pair_name}")
print(f"{'='*80}")
print(f"  {'Session':>12} | {'Trades':>7} | {'Win Rate':>8} | {'Net PnL':>12} | "
      f"{'Avg PnL':>10} | {'PF':>6} | {'Avg Dur':>8}")
print(f"  {'-'*80}")
for session, r in session_stats.iterrows():
    print(f"  {session:>12} | {int(r['trades']):>7} | {r['win_rate']:>7.1%} | "
          f"${r['net_pnl']:>+11,.2f} | ${r['avg_pnl']:>+9,.2f} | "
          f"{r['profit_factor']:>6.2f} | {r['avg_dur_min']:>6.0f}min")
print(f"{'='*80}")

# Key AMD validation: what % of trades happen during NY Distribution?
ny_trades = len(trade_df[trade_df["session"] == "New York"])
ny_pct = ny_trades / max(len(trade_df), 1)
print(f"\n  AMD Validation: {ny_pct:.1%} of trades during NY session (Distribution)")
print(f"  Expected: >60% for proper AMD strategy execution")

## 10. Realistic Friction Simulation — What Would OANDA Actually Return?

Apply real-world OANDA friction: slippage, off-hours spread widening, news spikes, micro-lot size limits. Scalpers are hit harder by friction than swing traders.

In [ ]:
# ── Realistic Friction for $100 OANDA Scalping Account ─────────────────────────
np.random.seed(42)

# OANDA-specific friction — HIGHER for scalping (tighter targets = more friction impact)
SLIPPAGE_RANGE = (0.3, 2.0)
SPREAD_OFFHOURS = 1.5
SPREAD_NEWS_PROB = 0.05
SPREAD_NEWS_EXTRA = 4.0
LOT_CAP = 0.5
LATENCY_MISS_RATE = 0.02
PIP_VALUE = PIP_VALUES.get(PRIMARY_PAIR, 6.5)

print(f"{'='*70}")
print(f"  OANDA FRICTION SIMULATION — AMD Scalper")
print(f"{'='*70}")
print(f"  Account         : $100 OANDA (101-001-30902818-003)")
print(f"  Slippage        : {SLIPPAGE_RANGE[0]}-{SLIPPAGE_RANGE[1]} pips")
print(f"  Off-hours spread: +{SPREAD_OFFHOURS} pips")
print(f"  News spike      : {SPREAD_NEWS_PROB:.0%} at +{SPREAD_NEWS_EXTRA} pips")
print(f"  Lot cap         : {LOT_CAP} lots max")
print(f"  Latency misses  : {LATENCY_MISS_RATE:.0%}")
print(f"{'='*70}")

realistic_rows = []
balance_theo = bt_config.initial_balance
balance_real = bt_config.initial_balance

for _, t in trade_df.iterrows():
    balance_theo += t["pnl"]

    if np.random.random() < LATENCY_MISS_RATE:
        realistic_rows.append({
            "trade_num": t["trade_num"], "original_pnl": t["pnl"],
            "adjusted_pnl": 0, "skipped": True,
            "slippage_cost": 0, "spread_cost": 0, "lot_penalty": 0,
            "balance_theo": balance_theo, "balance_real": balance_real,
        })
        continue

    slippage = np.random.uniform(*SLIPPAGE_RANGE)
    extra_spread = 0.0
    hour = t.get("entry_hour", 12)
    if hour < 7 or hour >= 21:
        extra_spread += SPREAD_OFFHOURS
    if np.random.random() < SPREAD_NEWS_PROB:
        extra_spread += SPREAD_NEWS_EXTRA

    lot_ratio = min(1.0, LOT_CAP / max(t["size_lots"], 0.001))
    friction_cost = (slippage + extra_spread) * PIP_VALUE * min(t["size_lots"], LOT_CAP)
    adjusted_pnl = (t["pnl"] * lot_ratio) - friction_cost
    balance_real += adjusted_pnl

    realistic_rows.append({
        "trade_num": t["trade_num"], "original_pnl": t["pnl"],
        "adjusted_pnl": adjusted_pnl, "skipped": False,
        "slippage_cost": slippage * PIP_VALUE * min(t["size_lots"], LOT_CAP),
        "spread_cost": extra_spread * PIP_VALUE * min(t["size_lots"], LOT_CAP),
        "lot_penalty": t["pnl"] * (1 - lot_ratio),
        "balance_theo": balance_theo, "balance_real": balance_real,
    })

rf = pd.DataFrame(realistic_rows)
active = rf[~rf["skipped"]]

theo_roi = (balance_theo / bt_config.initial_balance - 1) * 100
real_roi = (balance_real / bt_config.initial_balance - 1) * 100
real_wins = (active["adjusted_pnl"] > 0).sum()
real_total = len(active)
real_wr = real_wins / max(real_total, 1)
real_gp = active.loc[active["adjusted_pnl"] > 0, "adjusted_pnl"].sum()
real_gl = abs(active.loc[active["adjusted_pnl"] <= 0, "adjusted_pnl"].sum())
real_pf = real_gp / max(real_gl, 1e-9)

test_months = max(1, (trade_df["exit_time"].max() - trade_df["entry_time"].min()).days / 30)
real_annual = ((balance_real / bt_config.initial_balance) ** (12 / test_months) - 1) * 100

print(f"\n{'='*70}")
print(f"  {'METRIC':<25} {'THEORETICAL':>15} {'REALISTIC':>15}")
print(f"{'='*70}")
print(f"  {'Trades Executed':<25} {len(trade_df):>15,} {real_total:>15,}")
print(f"  {'Trades Skipped':<25} {'0':>15} {rf['skipped'].sum():>15,}")
print(f"  {'Final Balance':<25} ${balance_theo:>14,.2f} ${balance_real:>14,.2f}")
print(f"  {'Total ROI':<25} {theo_roi:>+14.1f}% {real_roi:>+14.1f}%")
print(f"  {'Annualized ROI':<25} {'—':>15} {real_annual:>+14.1f}%")
print(f"  {'Win Rate':<25} {results.win_rate:>14.1%} {real_wr:>14.1%}")
print(f"  {'Profit Factor':<25} {results.profit_factor:>15.2f} {real_pf:>15.2f}")
print(f"  {'Total Friction Cost':<25} {'$0':>15} "
      f"${(active['slippage_cost'].sum() + active['spread_cost'].sum()):>14,.2f}")
print(f"{'='*70}")

# Compound projections
print(f"\n  Projections from $100 at {real_annual:+.0f}% annualized:")
for yr in [1, 2, 3, 5]:
    projected = bt_config.initial_balance * (1 + real_annual / 100) ** yr
    print(f"    Year {yr}: ${projected:>12,.2f}")

# Equity curves — Theoretical vs Realistic
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(rf["trade_num"], rf["balance_theo"], lw=1.2, color="#2196F3", alpha=0.6, label="Theoretical")
axes[0].plot(rf["trade_num"], rf["balance_real"], lw=1.5, color="#4CAF50", label="Realistic (OANDA)")
axes[0].axhline(bt_config.initial_balance, ls="--", color="gray", alpha=0.5)
axes[0].set_ylabel("Balance ($)")
axes[0].set_title("Theoretical vs Realistic Equity — AMD Scalper on OANDA")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(rf["trade_num"], rf["balance_theo"], lw=1.2, color="#2196F3", alpha=0.6, label="Theoretical")
axes[1].plot(rf["trade_num"], rf["balance_real"], lw=1.5, color="#4CAF50", label="Realistic")
axes[1].set_yscale("log")
axes[1].set_xlabel("Trade #")
axes[1].set_ylabel("Balance ($) — log")
axes[1].set_title("Log Scale (straight = consistent compounding)")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 11. Save All Results to Google Drive

In [ ]:
# ── Save everything ───────────────────────────────────────────────────────────
SAVE_DIR = DRIVE_ROOT / "backtest_results" / CKPT_DIR.name
os.makedirs(SAVE_DIR, exist_ok=True)

# Trade log
trade_export = trade_df.drop(
    columns=["exit_date", "exit_week", "exit_month", "exit_year", "streak",
             "entry_hour", "entry_dow", "session"],
    errors="ignore",
)
trade_export.to_csv(SAVE_DIR / "trade_log.csv", index=False)
print(f"Saved trade_log.csv           ({len(trade_export)} trades)")

# Period breakdowns
daily.to_csv(SAVE_DIR / "daily_breakdown.csv", index=False)
print(f"Saved daily_breakdown.csv     ({len(daily)} days)")

weekly_export = weekly.copy()
weekly_export["exit_week"] = weekly_export["exit_week"].astype(str)
weekly_export.to_csv(SAVE_DIR / "weekly_breakdown.csv", index=False)
print(f"Saved weekly_breakdown.csv    ({len(weekly_export)} weeks)")

monthly_export = monthly.copy()
monthly_export["exit_month"] = monthly_export["exit_month"].astype(str)
monthly_export.to_csv(SAVE_DIR / "monthly_breakdown.csv", index=False)
print(f"Saved monthly_breakdown.csv   ({len(monthly_export)} months)")

yearly.to_csv(SAVE_DIR / "yearly_breakdown.csv", index=False)
print(f"Saved yearly_breakdown.csv    ({len(yearly)} years)")

# Equity curve
eq_df = pd.DataFrame({"equity": results.equity_curve})
eq_df.to_csv(SAVE_DIR / "equity_curve.csv", index=False)
print(f"Saved equity_curve.csv        ({len(eq_df)} points)")

# Session stats
session_stats.to_csv(SAVE_DIR / "session_breakdown.csv")
print(f"Saved session_breakdown.csv   ({len(session_stats)} sessions)")

# Friction simulation
rf.to_csv(SAVE_DIR / "friction_simulation.csv", index=False)
print(f"Saved friction_simulation.csv ({len(rf)} rows)")

# Per-pair aggregate summary
agg_rows = []
for pt in all_results:
    r, bc = all_results[pt]
    agg_rows.append({
        "pair": PAIR_NAMES[pt], "trades": r.total_trades,
        "win_rate": r.win_rate, "profit_factor": r.profit_factor,
        "final_balance": r.final_balance,
        "roi_pct": (r.final_balance / bc.initial_balance - 1) * 100,
        "max_drawdown": r.max_drawdown, "sharpe": r.sharpe_ratio,
    })
pd.DataFrame(agg_rows).to_csv(SAVE_DIR / "per_pair_summary.csv", index=False)
print(f"Saved per_pair_summary.csv    ({len(agg_rows)} pairs)")

print(f"\nAll results saved to: {SAVE_DIR}")
for f_name in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(SAVE_DIR / f_name)
    print(f"  {f_name:<35s}  {size/1024:>8.1f} KB")

## 12. Equity Curve & Sample Trades

In [ ]:
# ── Equity Curve & Sample Trades ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
equity = results.equity_curve

ax.plot(equity, lw=1.2, color="#4CAF50")
ax.axhline(bt_config.initial_balance, ls="--", color="gray", alpha=0.7, label="$100 start")
ax.fill_between(range(len(equity)), bt_config.initial_balance, equity,
                where=[e > bt_config.initial_balance for e in equity],
                color="#4CAF50", alpha=0.15)
ax.fill_between(range(len(equity)), bt_config.initial_balance, equity,
                where=[e < bt_config.initial_balance for e in equity],
                color="#F44336", alpha=0.15)
ax.set_title(f"AMD Scalper Equity Curve — {pair_name} ($100 start, 10% risk, OANDA)")
ax.set_xlabel("Bars")
ax.set_ylabel("Balance ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSample Trades (first 15):")
print(f"  {'Dir':>5}  {'Entry':>10}  {'Exit':>10}  {'PnL':>10}  {'Reason':>14}  {'Win':>4}  {'Dur':>8}")
print(f"  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*14}  {'─'*4}  {'─'*8}")
for _, row in trade_df.head(15).iterrows():
    win = "✓" if row["is_winner"] else "✗"
    dur = f"{row['duration'].total_seconds()/60:.0f}min"
    print(f"  {row['direction']:>5}  {row['entry_price']:>10.3f}  {row['exit_price']:>10.3f}  "
          f"${row['pnl']:>+9.2f}  {row['exit_reason']:>14}  {win:>4}  {dur:>8}")